In [331]:
import pygeoinf as inf 
import numpy as np
from typing import List, Any
import matplotlib.pyplot as plt

class GridSpace(inf.HilbertSpace):
    def __init__(self, shape: tuple[int, ...]):
        self._shape = shape
        self._dim = np.prod(shape)
        
    @property
    def dim(self) -> int:
        return self._dim

    @property
    def shape(self) -> tuple[int, ...]:
        return self._shape

    def to_components(self, x: np.ndarray) -> np.ndarray:
        return x.flatten()

    def from_components(self, c: np.ndarray) -> np.ndarray:
        return c.reshape(self.shape)
    
    def add(self, x: np.ndarray, y: np.ndarray) -> np.ndarray: # todo: should this be necessary?
        # Ensure both operands are in the same shape
        if x.shape != y.shape:
            # Convert both to component form, add, then convert back
            x_flat = self.to_components(x) if x.shape == self.shape else x
            y_flat = self.to_components(y) if y.shape == self.shape else y
            result_flat = x_flat + y_flat
            return self.from_components(result_flat)
        return x + y

    def to_dual(self, x: np.ndarray) -> "inf.LinearForm":
        return inf.LinearForm(self, components=self.to_components(x))

    def from_dual(self, xp: "inf.LinearForm") -> np.ndarray:
        return self.from_components(xp.components)

    def __eq__(self, other: object):
        if not isinstance(other, GridSpace):
            return NotImplemented
        return self.shape == other.shape

s = GridSpace((12, 8))
x = s.random()
y = s.random()

inner_prod = s.inner_product(x, y)
norm_x = s.norm(x)

print(f"The grid space has dimension: {s.dim}")
print(f"A random vector x: {x}")
print(f"The inner product <x, y> is: {inner_prod:.4f}")

The grid space has dimension: 96
A random vector x: [[ 1.22886037e+00  4.93015943e-01 -1.04340516e+00  5.99249304e-02
  -1.06715281e-01 -1.73636081e-01  1.20493710e+00  3.53534783e-01]
 [-4.45902825e-01  1.43936521e+00 -9.13021090e-01  9.30595518e-01
   5.68556510e-01  3.60930380e-01 -1.08948829e+00 -7.36417089e-02]
 [-6.07936758e-01 -1.04233631e+00  2.61064728e-01  1.48270825e+00
   4.86227381e-01 -2.32969622e-01 -1.70264584e-01  2.50905788e-01]
 [ 3.40321694e-01 -3.89637594e-01  5.92644807e-01  8.49138270e-01
  -1.18213679e+00  1.59966475e+00  1.14264217e+00  9.45369473e-01]
 [ 7.04498055e-01  9.77409954e-01  5.49269357e-02  1.39485588e-01
  -3.66469309e-01  2.72389546e+00 -3.35722540e-01  5.00820502e-01]
 [-2.81673488e-01 -7.64148992e-01 -2.81468156e-01 -1.90576097e+00
   1.11417812e+00 -5.54175548e-01  6.39179960e-01  4.49910003e-01]
 [-1.41422925e+00  2.19488697e-01  7.56818373e-01 -1.80528459e+00
   6.39101216e-01  1.37868338e-01  1.53088622e+00 -7.18607657e-01]
 [ 8.78355094e-01

In [332]:
mat = np.array([[1, 2, 3, 4],
                [5, 6, 7, 8],
                [1, 1, 1, 1]])

class WaterMassForwardProblem:
    def __init__(self, space: GridSpace, mat: np.ndarray):
        self._space = space
        self._mat = mat
        self._number_of_observation_variables = mat.shape[0]
        self._number_of_water_masses = mat.shape[1]

    @property
    def space(self) -> GridSpace:
        return self._space

    @property
    def mat(self) -> np.ndarray:
        return self._mat

    def form_stacked_vector(self, x: List[np.ndarray]) -> np.ndarray:
        # Use space's to_components method instead of flatten
        return np.column_stack([self.space.to_components(arr) for arr in x])
    
    def apply_matrix(self, v: np.ndarray) -> np.ndarray:
        # Ensure the matrix and vector dimensions are compatible
        assert self.mat.shape[1] == v.shape[1], "Matrix columns must match number of input variables"
        
        # Perform the matrix multiplication
        result = self.mat @ v.T  # Shape will be (m, n)
        return result.T  # Return shape (n, m)
    
    def separate_stacked_vector(self, v: np.ndarray) -> List[np.ndarray]:
        # Use space's from_components method instead of reshape
        return [self.space.from_components(v[:, i]) for i in range(v.shape[1])]
    
    def transform(self, x: List[np.ndarray]) -> List[np.ndarray]:
        # Validate input length matches matrix expectation
        assert len(x) == self._number_of_water_masses, f"Expected {self._number_of_water_masses} input arrays, got {len(x)}"

        # Combine all operations in one method
        stacked = self.form_stacked_vector(x)
        transformed = self.apply_matrix(stacked)
        return self.separate_stacked_vector(transformed)
    
    def as_linear_operator(self) -> inf.LinearOperator:
        # Create product space for multiple inputs
        input_space = inf.HilbertSpaceDirectSum([self.space] * self._number_of_water_masses)
        output_space = inf.HilbertSpaceDirectSum([self.space] * self._number_of_observation_variables)

        def forward_mapping(x_list):
            return self.transform(x_list)
        
        return inf.LinearOperator(
            input_space,
            output_space,
            forward_mapping
        )

In [335]:
def create_spatial_covariance_matrix(shape: tuple[int, ...], std: float, length_scale: float) -> np.ndarray:
    """
    Create a covariance matrix with squared exponential (Gaussian) decay based on spatial distance.
    
    Args:
        shape: Grid shape (e.g., (4, 5) for 2D grid)
        std: Standard deviation for all grid points  
        length_scale: Length scale for squared exponential decay
    
    Returns:
        Covariance matrix of size (prod(shape), prod(shape))
    """
    # Get total number of grid points
    n_points = np.prod(shape)
    
    # Create coordinate arrays for each grid point
    if len(shape) == 2:
        rows, cols = shape
        coords = np.array([[i, j] for i in range(rows) for j in range(cols)])
    elif len(shape) == 3:
        d1, d2, d3 = shape
        coords = np.array([[i, j, k] for i in range(d1) for j in range(d2) for k in range(d3)])
    else:
        # General case for arbitrary dimensions
        indices = np.meshgrid(*[range(dim) for dim in shape], indexing='ij')
        coords = np.column_stack([idx.ravel() for idx in indices])
    
    # Initialize covariance matrix
    cov_matrix = np.zeros((n_points, n_points))
    
    # Fill covariance matrix
    for i in range(n_points):
        for j in range(n_points):
            # Calculate Euclidean distance between grid points
            distance = np.linalg.norm(coords[i] - coords[j])
            
            # Squared exponential (Gaussian) covariance
            if i == j:
                # Diagonal elements (variance)
                cov_matrix[i, j] = std**2
            else:
                # Off-diagonal elements (squared exponential decay)
                cov_matrix[i, j] = std**2 * np.exp(-(distance**2) / (2 * length_scale**2))
    
    return cov_matrix

In [336]:
problem = WaterMassForwardProblem(s, mat)
forward_operator = problem.as_linear_operator()

data_error_measure = inf.GaussianMeasure.from_standard_deviation(s, 0.001)
joint_data_error_measure = inf.GaussianMeasure.from_direct_sum([data_error_measure] * mat.shape[0])

forward_problem = inf.LinearForwardProblem(forward_operator, data_error_measure=joint_data_error_measure)

In [328]:
prior_covariance_matrix = create_spatial_covariance_matrix(s.shape, 0.3, 2.0)
prior_expectation = np.full(s.shape,  0.5)
prior_measure = inf.GaussianMeasure.from_covariance_matrix(s, prior_covariance_matrix, expectation=prior_expectation)
joint_prior_measure = inf.GaussianMeasure.from_direct_sum([prior_measure] * mat.shape[1])

true_model, data = forward_problem.synthetic_model_and_data(joint_prior_measure)

inversion = inf.LinearBayesianInversion(forward_problem, prior_measure)
solver = inf.CholeskySolver(galerkin=True)
posterior = inversion.model_posterior_measure(data, solver)

AttributeError: 'list' object has no attribute 'flatten'

In [330]:
inversion.normal_operator.matrix(dense=True)

AttributeError: 'list' object has no attribute 'flatten'

In [ ]:
# Plot the results - each row is a water mass, and show true vs posterior mean
for i in range(mat.shape[1]):
    plt.figure(figsize=(10, 4))
    
    # True model
    plt.subplot(1, 2, 1)
    plt.title(f"True Model - Water Mass {i+1}")
    plt.imshow(true_model[i], cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Value')
    
    # Posterior mean
    plt.subplot(1, 2, 2)
    plt.title(f"Posterior Mean - Water Mass {i+1}")
    plt.imshow(posterior.expectation[i], cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Value')
    
    plt.tight_layout()